# 03 - ALL CODE IN ORDER

**Course Work Nazih | M2AD Motor | Single-image binary classification**

Run every code cell **top-to-bottom** in Google Colab (GPU recommended).
Only **Normal (0)** and **Defective (1)** classes are used.

- Baseline: ResNet18 + Cross-Entropy.
- Paper idea: same ResNet18 + Poly-1 Loss (epsilon = 1).
- This run is a **development validation**, not an independent final test.
- Dataset images and training outputs remain in Colab; do **not** upload them to GitHub.


## 1. Setup, fixed settings, and historical manifest

The original 99-image manifest is embedded in this notebook; no separate manifest file is needed.
The manifest records original SHA256 hashes of the images and preserves the historical development split.


In [ ]:
import os, sys, io, json, csv, time, zlib, struct, random, zipfile, hashlib, platform, urllib.request, argparse
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
from PIL import Image
import torch, torchvision
from torch import nn
from torch.nn import functional as F
from torchvision.models import resnet18, ResNet18_Weights

ROOT = Path('/content/nazih_motor_runtime') if Path('/content').is_dir() else Path.cwd() / 'nazih_motor_runtime'
ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / 'resnet18_supervised').mkdir(exist_ok=True)
MANIFEST_TEXT = "[\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/009/A000_I01.png\",\n    \"object\": \"009\",\n    \"group\": \"009\",\n    \"label\": 0,\n    \"sha256\": \"54ff5c2249feee6fc60d9c47b1365b7df9304b19a7ca9d3d0f53f9317465e18f\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/020/A000_I01.png\",\n    \"object\": \"020\",\n    \"group\": \"020\",\n    \"label\": 0,\n    \"sha256\": \"24841c8df60780fcc6a9e9a046eebfc3d881173fbb7de64a297766e5ee1147db\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/029/A000_I01.png\",\n    \"object\": \"029\",\n    \"group\": \"029\",\n    \"label\": 0,\n    \"sha256\": \"120f4bfecf3f217709a96ea55a2a1b36ebe8ac21435163fdfb95bff977c6c2e1\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/028/A000_I01.png\",\n    \"object\": \"028\",\n    \"group\": \"028\",\n    \"label\": 0,\n    \"sha256\": \"3b3d248049d2cc5ad1581ad91a10e243b70a5b6a3f1ccd1b88456c77a80ddc89\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/011/A000_I01.png\",\n    \"object\": \"011\",\n    \"group\": \"011\",\n    \"label\": 0,\n    \"sha256\": \"4b7215d84ab6dd1545d5fb2bf467e7f11bc96be1c9650658d8f0b52274faaa9f\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/000/A000_I01.png\",\n    \"object\": \"000\",\n    \"group\": \"000\",\n    \"label\": 0,\n    \"sha256\": \"e57989bf9d04278d0fe0369c4b946dad4e7030c1aa05ac819c67a1987a53f75e\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/016/A000_I01.png\",\n    \"object\": \"016\",\n    \"group\": \"016\",\n    \"label\": 0,\n    \"sha256\": \"b420fdc704d2747f7d2dcbb14cb004920c230e6552e6cf560b0a260b9352b5a8\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/042/A000_I01.png\",\n    \"object\": \"042\",\n    \"group\": \"042\",\n    \"label\": 0,\n    \"sha256\": \"07181a26f0973288507a8c6837d4984196ff97a6ac85b2646cdb67b63b04471c\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/048/A000_I01.png\",\n    \"object\": \"048\",\n    \"group\": \"048\",\n    \"label\": 0,\n    \"sha256\": \"a983f6406a7847e5ca328f062fb52c5fb233a7c3418aa0e8cf8013dd20a98110\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/024/A000_I01.png\",\n    \"object\": \"024\",\n    \"group\": \"024\",\n    \"label\": 0,\n    \"sha256\": \"e7b7ad05e7a676b51503468762046916b9f3afb3505a59100eaa92a27bd3ba44\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/031/A000_I01.png\",\n    \"object\": \"031\",\n    \"group\": \"031\",\n    \"label\": 0,\n    \"sha256\": \"0da6d44babe4295869eea8273c6def4f8bf84a0dbb216e99d4f902cba0db4d48\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/041/A000_I01.png\",\n    \"object\": \"041\",\n    \"group\": \"041\",\n    \"label\": 0,\n    \"sha256\": \"acf32558ec0c31d7e8e5e69f442e8b72731e9431fa7181636461e886a6ac8fe3\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/033/A000_I01.png\",\n    \"object\": \"033\",\n    \"group\": \"033\",\n    \"label\": 0,\n    \"sha256\": \"dfea3daf54b875ec60ad71137f1dfccddc6fe722b91b629dcd632ea6232bf609\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/004/A000_I01.png\",\n    \"object\": \"004\",\n    \"group\": \"004\",\n    \"label\": 0,\n    \"sha256\": \"e08d8fbec1d5830d809e3d0a4e1aadfcde4d5aea5063a2ff1a8f5e967f041a50\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/030/A000_I01.png\",\n    \"object\": \"030\",\n    \"group\": \"030\",\n    \"label\": 0,\n    \"sha256\": \"31551e5adbbb64976293b310946e1b0fbf084040e10d0c929467f5874933577b\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/010/A000_I01.png\",\n    \"object\": \"010\",\n    \"group\": \"010\",\n    \"label\": 0,\n    \"sha256\": \"53f3216279cdc6888784be805566e155cd2cb4b51e9118e890027594faaf024c\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/038/A000_I01.png\",\n    \"object\": \"038\",\n    \"group\": \"038\",\n    \"label\": 0,\n    \"sha256\": \"ecfcd03be4c32ca659b16da2237f5e6b4fc3cd9b14df8122001f2788801e945d\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/012/A000_I01.png\",\n    \"object\": \"012\",\n    \"group\": \"012\",\n    \"label\": 0,\n    \"sha256\": \"775ea9ad0a1024b1a53ba224ac053d479b6abf5e9a2f5c5a16ee36627362dafd\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/021/A000_I01.png\",\n    \"object\": \"021\",\n    \"group\": \"021\",\n    \"label\": 0,\n    \"sha256\": \"a08d65ca7c1282de909244a2b97440dbf5beec435205a28687488bfb73ee9313\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/035/A000_I01.png\",\n    \"object\": \"035\",\n    \"group\": \"035\",\n    \"label\": 0,\n    \"sha256\": \"0c2279e2f78a3ea91a1521066e47f70ac8394c81caa424628a2081f263613f46\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/019/A000_I01.png\",\n    \"object\": \"019\",\n    \"group\": \"019\",\n    \"label\": 0,\n    \"sha256\": \"c82de723623f56b11f08c1dda016e541eec2e6fad62410a8c0681f24c0655d37\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/045/A000_I01.png\",\n    \"object\": \"045\",\n    \"group\": \"045\",\n    \"label\": 0,\n    \"sha256\": \"7e0f961cf63538678929cef03c6cdc53e1269f0494507cdb8bcec2f828e5745d\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/025/A000_I01.png\",\n    \"object\": \"025\",\n    \"group\": \"025\",\n    \"label\": 0,\n    \"sha256\": \"6c5d35c303404347de29fc5331ea296e3647b22a25c795539855f90b3e612cbd\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_comparison/data/Motor/Good/026/A000_I01.png\",\n    \"object\": \"026\",\n    \"group\": \"026\",\n    \"label\": 0,\n    \"sha256\": \"760dcba6d2403fcaa7a355a0c174b43564a0d20fa721c624fc532609d73db567\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_classification/data/Motor/Good/018/A000_I01.png\",\n    \"object\": \"018\",\n    \"group\": \"018\",\n    \"label\": 0,\n    \"sha256\": \"b0b77e5b1d0f41b1dd7d953e6d1cc61ab44392855ca8e6dc31fad45654737e56\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_classification/data/Motor/Good/003/A000_I01.png\",\n    \"object\": \"003\",\n    \"group\": \"003\",\n    \"label\": 0,\n    \"sha256\": \"0180c8c1175a6ebfb0afce4116118def8cda4f6644ca89720e59665411118b46\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_classification/data/Motor/Good/023/A000_I01.png\",\n    \"object\": \"023\",\n    \"group\": \"023\",\n    \"label\": 0,\n    \"sha256\": \"b391c9b12462a575dbafbae5484494997b7a884dd32fa9e56c140cebc750b411\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_classification/data/Motor/Good/039/A000_I01.png\",\n    \"object\": \"039\",\n    \"group\": \"039\",\n    \"label\": 0,\n    \"sha256\": \"e9fa0fe5baeeb214d5648be8c109e6ebd40e74414a5838d190295b4fc31c75b9\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_classification/data/Motor/Good/022/A000_I01.png\",\n    \"object\": \"022\",\n    \"group\": \"022\",\n    \"label\": 0,\n    \"sha256\": \"6e861ed774cdb92b30250358893c7f2d1e114b34454f6b4de65b5b1cbf781e38\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_classification/data/Motor/Good/043/A000_I01.png\",\n    \"object\": \"043\",\n    \"group\": \"043\",\n    \"label\": 0,\n    \"sha256\": \"e6c61ad031ec24c18ff982eb3b2f28e02a0e64014a61533f454104fb32880a73\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/001/A000_I01.png\",\n    \"object\": \"001\",\n    \"group\": \"001\",\n    \"label\": 0,\n    \"sha256\": \"88b63a4fae3197cad0a79973b1df2205db0b2d60c31a6664f0a10c8828cdc3fa\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/002/A000_I01.png\",\n    \"object\": \"002\",\n    \"group\": \"002\",\n    \"label\": 0,\n    \"sha256\": \"bbe3923a3e0284ca698c8bff16052121f3189268987c3cfb03f978b96c78314c\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/005/A000_I01.png\",\n    \"object\": \"005\",\n    \"group\": \"005\",\n    \"label\": 0,\n    \"sha256\": \"a363b17660b3433628dda14470c2b842840ddaac099d44093c4f602ae7da0f46\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/006/A000_I01.png\",\n    \"object\": \"006\",\n    \"group\": \"006\",\n    \"label\": 0,\n    \"sha256\": \"a7d68cadcdafff048a37b0832919550ada7cf8ac71c99065f4774e526e91fc81\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/007/A000_I01.png\",\n    \"object\": \"007\",\n    \"group\": \"007\",\n    \"label\": 0,\n    \"sha256\": \"b2a7bcc22122f82531a33c60b8660f04ad1ef02108f7328037f4fb58dd55d53c\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/008/A000_I01.png\",\n    \"object\": \"008\",\n    \"group\": \"008\",\n    \"label\": 0,\n    \"sha256\": \"054e60da39783b3cd257862b7027bb0928ffd0eac7c62f9f8afd219e2ea201c7\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/013/A000_I01.png\",\n    \"object\": \"013\",\n    \"group\": \"013\",\n    \"label\": 0,\n    \"sha256\": \"298c7ebdd987670a88b39b4bd9814ac59d223cbde75f70dc76d6306966b2fb0b\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/014/A000_I01.png\",\n    \"object\": \"014\",\n    \"group\": \"014\",\n    \"label\": 0,\n    \"sha256\": \"3ba3940472d604128a2c73843ce6d9a2874e63678d973e65c6fdad19e6e506f8\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/015/A000_I01.png\",\n    \"object\": \"015\",\n    \"group\": \"015\",\n    \"label\": 0,\n    \"sha256\": \"04ac4c1f6092dc444e063bc76284aa6cf2557f6548a1c8fcf14b15e7d48aa74a\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/017/A000_I01.png\",\n    \"object\": \"017\",\n    \"group\": \"017\",\n    \"label\": 0,\n    \"sha256\": \"6e992c06cd626d53b33587d21647a3eda386deec440958a17be6967be0c20453\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/027/A000_I01.png\",\n    \"object\": \"027\",\n    \"group\": \"027\",\n    \"label\": 0,\n    \"sha256\": \"b7ca0f30e68c9905bd909d2fefed881ee517d2ec48d8a0b82f0c262a6f70c892\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/032/A000_I01.png\",\n    \"object\": \"032\",\n    \"group\": \"032\",\n    \"label\": 0,\n    \"sha256\": \"d1d2c42b3b308ef7fdc898fdcd30a45f5beb99f62585ba313acfa19c0c144f43\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/034/A000_I01.png\",\n    \"object\": \"034\",\n    \"group\": \"034\",\n    \"label\": 0,\n    \"sha256\": \"82bad0819da25b891b00896ec7fe73f7c75af096ef25f4c26f85e5df5a0cee7a\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/036/A000_I01.png\",\n    \"object\": \"036\",\n    \"group\": \"036\",\n    \"label\": 0,\n    \"sha256\": \"4e2af871bfc39792ec87d6e905ba9ad537c2e48ebbcf6525ae8f4065392feb76\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/037/A000_I01.png\",\n    \"object\": \"037\",\n    \"group\": \"037\",\n    \"label\": 0,\n    \"sha256\": \"438eb3e64b06dd1ebe566cd08a4f5dc74c3a8fa458e9ea2ec9759569d5f29699\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/040/A000_I01.png\",\n    \"object\": \"040\",\n    \"group\": \"040\",\n    \"label\": 0,\n    \"sha256\": \"014c4ab5c3ddcb485c30803c3efbce08c2bce5ba1f7187f953c775f074c75b75\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/044/A000_I01.png\",\n    \"object\": \"044\",\n    \"group\": \"044\",\n    \"label\": 0,\n    \"sha256\": \"efabbae3e2a72aec063c61e9437d5e2cd949a5726937394f431feb48bbc5ddc5\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/046/A000_I01.png\",\n    \"object\": \"046\",\n    \"group\": \"046\",\n    \"label\": 0,\n    \"sha256\": \"38855335b54e325c50d16be4e6ddb8412ccdcca648e29651f5a0bf7c1c2fbea0\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/047/A000_I01.png\",\n    \"object\": \"047\",\n    \"group\": \"047\",\n    \"label\": 0,\n    \"sha256\": \"638a57273f1deeab0e178c81c7a5054cd3d77bfb9e159c6e76a6a6a6fc7e1292\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/Good/049/A000_I01.png\",\n    \"object\": \"049\",\n    \"group\": \"049\",\n    \"label\": 0,\n    \"sha256\": \"b50ef3df55bb14ddbe7bc32659965041159b7abba22f64373bbb6c7c6584db7b\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_1_000/A000_I01.png\",\n    \"object\": \"hole_1_000\",\n    \"group\": \"000\",\n    \"label\": 1,\n    \"sha256\": \"f8708dc4993954a188d7856b2b35f918249e3c5f024e187a69180c180484f0ee\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_1_001/A000_I01.png\",\n    \"object\": \"hole_1_001\",\n    \"group\": \"001\",\n    \"label\": 1,\n    \"sha256\": \"3275c475ad14f3c941a77e048c7fcde0d66e244c79bce75a9fe2c92ce27f7298\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_1_002/A000_I01.png\",\n    \"object\": \"hole_1_002\",\n    \"group\": \"002\",\n    \"label\": 1,\n    \"sha256\": \"db1286385d8cdb5c8472d4251a195f067e941a09caeb500421929e9e3eb33853\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_1_003/A000_I01.png\",\n    \"object\": \"hole_1_003\",\n    \"group\": \"003\",\n    \"label\": 1,\n    \"sha256\": \"8d76545e790b57288497e7f16a20fa4356797423d217bb43fe3bc2e8a8e55ea0\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_1_004/A000_I01.png\",\n    \"object\": \"hole_1_004\",\n    \"group\": \"004\",\n    \"label\": 1,\n    \"sha256\": \"5c772cb4d2beb9d18cb3214fd6c142940bcadbbbd7d1bb82f00201f019dda019\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_005/A000_I01.png\",\n    \"object\": \"damage_1_005\",\n    \"group\": \"005\",\n    \"label\": 1,\n    \"sha256\": \"a43542a572c8db7422320b4904e93c8b79d47d31c8caadaa17ace1efc70e462b\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_006/A000_I01.png\",\n    \"object\": \"damage_1_006\",\n    \"group\": \"006\",\n    \"label\": 1,\n    \"sha256\": \"0b403c5e2c9ca2bc186cdd254696bf4975f48efb79f78d94cf64192955fd58df\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_007/A000_I01.png\",\n    \"object\": \"damage_1_007\",\n    \"group\": \"007\",\n    \"label\": 1,\n    \"sha256\": \"ee1cc486a5a00f21784c68bf68935216054c2a3a817b5012fd0c0f812282aaa1\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_008/A000_I01.png\",\n    \"object\": \"damage_1_008\",\n    \"group\": \"008\",\n    \"label\": 1,\n    \"sha256\": \"51d8d737fa29b6a55589e097f9814ded710c025e64a776241cf56df17140da4b\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_009/A000_I01.png\",\n    \"object\": \"damage_1_009\",\n    \"group\": \"009\",\n    \"label\": 1,\n    \"sha256\": \"c871a9be768d112b02d9fc92d61cf410a035344ca66a6826c3c232ac158d0d31\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_2_010/A000_I01.png\",\n    \"object\": \"hole_2_010\",\n    \"group\": \"010\",\n    \"label\": 1,\n    \"sha256\": \"3325ed80629c1052b0d6f6e42d43c5d681e72eabbb063fa6685aae1532fdfa32\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_2_011/A000_I01.png\",\n    \"object\": \"hole_2_011\",\n    \"group\": \"011\",\n    \"label\": 1,\n    \"sha256\": \"570862d535fc87b242932073101f9e83d79d6411de00883a968b37bfb7ded801\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_2_012/A000_I01.png\",\n    \"object\": \"hole_2_012\",\n    \"group\": \"012\",\n    \"label\": 1,\n    \"sha256\": \"339ebbd5d138ced8a89abee99d6822de4a4cf40cfde229e85844d56e86e07783\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_2_013/A000_I01.png\",\n    \"object\": \"hole_2_013\",\n    \"group\": \"013\",\n    \"label\": 1,\n    \"sha256\": \"188fef91236080557d499c2a0df16ab8e03dd63cdc1b05d79d1d33cb9a63d8e6\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/hole_3_014/A000_I01.png\",\n    \"object\": \"hole_3_014\",\n    \"group\": \"014\",\n    \"label\": 1,\n    \"sha256\": \"7f5e847ee9c98d0fc419ffee075f8186ba4f2f427111c12858663582eb2266d0\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_015/A000_I01.png\",\n    \"object\": \"damage_2_015\",\n    \"group\": \"015\",\n    \"label\": 1,\n    \"sha256\": \"86b6089e5a13caee9756aef6dee8b1adcb2e1d3bd55498ccab76beee3e053270\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_016/A000_I01.png\",\n    \"object\": \"damage_2_016\",\n    \"group\": \"016\",\n    \"label\": 1,\n    \"sha256\": \"9c52921e88ae362454f41adf9f15b43eb3ec1eb0e467ade980c9aa3b46cfac75\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_017/A000_I01.png\",\n    \"object\": \"damage_2_017\",\n    \"group\": \"017\",\n    \"label\": 1,\n    \"sha256\": \"81bdf92f3813a1ea3420b9ac7c6e6eb70dc11f02d9cecc160a6c8177be9677d5\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_018/A000_I01.png\",\n    \"object\": \"damage_2_018\",\n    \"group\": \"018\",\n    \"label\": 1,\n    \"sha256\": \"46c63e65f56ed46acc6ce04050d83530270187355222a4e9c234a5258fdeb4b5\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_019/A000_I01.png\",\n    \"object\": \"damage_2_019\",\n    \"group\": \"019\",\n    \"label\": 1,\n    \"sha256\": \"4b95498ae4383b6fd93628e78a28ae2ff0f896b42a78d7ebccaa11e9063444f5\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_hole_1_020/A000_I01.png\",\n    \"object\": \"damage_1_hole_1_020\",\n    \"group\": \"020\",\n    \"label\": 1,\n    \"sha256\": \"41f437b38ec62a60e0fe9e9a72d4791c20cae045e0a4ae03959df0913ed39571\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_hole_1_021/A000_I01.png\",\n    \"object\": \"damage_1_hole_1_021\",\n    \"group\": \"021\",\n    \"label\": 1,\n    \"sha256\": \"6e86566f6a30651a1c675bbe65fbd92fd187211d024b2c674512aa25d7c113dc\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_hole_2_022/A000_I01.png\",\n    \"object\": \"damage_1_hole_2_022\",\n    \"group\": \"022\",\n    \"label\": 1,\n    \"sha256\": \"f4cb125432fb155dc64949143fdb4791646f2bc59b2630b6cfad6667226aaf42\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_hole_1_023/A000_I01.png\",\n    \"object\": \"damage_2_hole_1_023\",\n    \"group\": \"023\",\n    \"label\": 1,\n    \"sha256\": \"faa4bbb107d8e2706aa8580d0ea509b8fe38854e0b5ca9dc6f9a81ac72020244\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_hole_2_024/A000_I01.png\",\n    \"object\": \"damage_2_hole_2_024\",\n    \"group\": \"024\",\n    \"label\": 1,\n    \"sha256\": \"5fc867f28f8f3aaeb05d1c94e801086867abd56edf0d41f6eb5dd7a5e4beb63a\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_025/A000_I01.png\",\n    \"object\": \"damage_2_025\",\n    \"group\": \"025\",\n    \"label\": 1,\n    \"sha256\": \"a3bd7e4d37f2743c61c2e482a3c10029677a22f15c5889158651c8a39b5a5b24\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_026/A000_I01.png\",\n    \"object\": \"damage_2_026\",\n    \"group\": \"026\",\n    \"label\": 1,\n    \"sha256\": \"c108d5935de46193e987b44a1affd23ad60b310bafd7bd846363ecb46703f09c\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_027/A000_I01.png\",\n    \"object\": \"damage_1_027\",\n    \"group\": \"027\",\n    \"label\": 1,\n    \"sha256\": \"bc2f5bd3c1ff1b85407e8ba894630faacf6c76248f263b54a45a0cc08cd94202\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_028/A000_I01.png\",\n    \"object\": \"damage_1_028\",\n    \"group\": \"028\",\n    \"label\": 1,\n    \"sha256\": \"76d73c342c9b5321454ac9e175dd18f67a57d5f12b1a1d33fa3e269c0b3a85f5\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_029/A000_I01.png\",\n    \"object\": \"damage_1_029\",\n    \"group\": \"029\",\n    \"label\": 1,\n    \"sha256\": \"65a3c39121ebb7ed3a5df3a533cc0d4fefab7434ee0bdb07cc0fb982a1b3879a\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_031/A000_I01.png\",\n    \"object\": \"damage_1_031\",\n    \"group\": \"031\",\n    \"label\": 1,\n    \"sha256\": \"3c1a2f3b954b2dea48b595ca0545c1badc2f62e65694cced268cd4f474872b95\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_032/A000_I01.png\",\n    \"object\": \"damage_2_032\",\n    \"group\": \"032\",\n    \"label\": 1,\n    \"sha256\": \"ce31b41f9e6e3b005abfd528b8e50c74ca195e080efe77ce05a82a499fb26542\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_1_033/A000_I01.png\",\n    \"object\": \"damage_1_033\",\n    \"group\": \"033\",\n    \"label\": 1,\n    \"sha256\": \"6969f8c7110967839f35c15b0649023287bdb008e1abfe85c530dbf757a037e7\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_2_034/A000_I01.png\",\n    \"object\": \"bending_2_034\",\n    \"group\": \"034\",\n    \"label\": 1,\n    \"sha256\": \"8fe7301fd0255451ede50ba215e247d81523172c0f056d6a0edb046379e9d77b\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_damage_1_035/A000_I01.png\",\n    \"object\": \"bending_1_damage_1_035\",\n    \"group\": \"035\",\n    \"label\": 1,\n    \"sha256\": \"17043224e86213069792a5df05cbf04f261086cfeaf8d0d344c6973042ec97c1\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_036/A000_I01.png\",\n    \"object\": \"bending_1_036\",\n    \"group\": \"036\",\n    \"label\": 1,\n    \"sha256\": \"d0fb4424fb8ba0459e16ff2fbc61043e4c8a5b6d221ba9ca9d5724895ca9f06c\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_damage_1_037/A000_I01.png\",\n    \"object\": \"bending_1_damage_1_037\",\n    \"group\": \"037\",\n    \"label\": 1,\n    \"sha256\": \"ea868c3abc1b412eec46ea9f2821a8ee8e1b88854cf0b8bce4c1cddd4e004c46\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_damage_1_038/A000_I01.png\",\n    \"object\": \"bending_1_damage_1_038\",\n    \"group\": \"038\",\n    \"label\": 1,\n    \"sha256\": \"67d950c887eb0a0ff830f8b6b94427ee9b80ae07bc526f5d6b4f743836e4bc77\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_2_039/A000_I01.png\",\n    \"object\": \"bending_2_039\",\n    \"group\": \"039\",\n    \"label\": 1,\n    \"sha256\": \"07bc40b10b96880fae92b020c9b48937413c3c4a433b7ca8ae78a851f48de327\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_damage_1_040/A000_I01.png\",\n    \"object\": \"bending_1_damage_1_040\",\n    \"group\": \"040\",\n    \"label\": 1,\n    \"sha256\": \"5d5a5669bbf42c546f2168d10d539b5eba55bac2f9ce80f19a195d6d8316ec2e\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_2_041/A000_I01.png\",\n    \"object\": \"bending_2_041\",\n    \"group\": \"041\",\n    \"label\": 1,\n    \"sha256\": \"f7c8826204481d9491d825d089796e76d152537ebe44d7179a5f17b945bcb8b3\",\n    \"split\": \"validation\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_042/A000_I01.png\",\n    \"object\": \"bending_1_042\",\n    \"group\": \"042\",\n    \"label\": 1,\n    \"sha256\": \"7c00636339a722d9ad60cf73002ccda45bb6cb993d4bdfa3cdfe71568686cc46\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_043/A000_I01.png\",\n    \"object\": \"damage_2_043\",\n    \"group\": \"043\",\n    \"label\": 1,\n    \"sha256\": \"5bad8f7e38aeb152c1800bf27e7d8847427709d19eadafaec1ed150af3a6ebc6\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_044/A000_I01.png\",\n    \"object\": \"bending_1_044\",\n    \"group\": \"044\",\n    \"label\": 1,\n    \"sha256\": \"9fef0ae68b6513344e853f35549dad880b7c6b3a096a4d760e481e4d8d87edd8\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_045/A000_I01.png\",\n    \"object\": \"bending_1_045\",\n    \"group\": \"045\",\n    \"label\": 1,\n    \"sha256\": \"620952efc343cac81ec0058292d752d5193aa292a6694969431052f3dc42d9b7\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/damage_2_046/A000_I01.png\",\n    \"object\": \"damage_2_046\",\n    \"group\": \"046\",\n    \"label\": 1,\n    \"sha256\": \"0ca961e5b3ee4cace95572b19877a235e38d370286a91f49fc8d9161a895e412\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_2_047/A000_I01.png\",\n    \"object\": \"bending_2_047\",\n    \"group\": \"047\",\n    \"label\": 1,\n    \"sha256\": \"52620b5fd14dfd7ce94961500506821858f05547a60391d83d549af31d323d8e\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_048/A000_I01.png\",\n    \"object\": \"bending_1_048\",\n    \"group\": \"048\",\n    \"label\": 1,\n    \"sha256\": \"60372de27bd233661dd200a793279b3e93360a12c0d5c9462250f1bbc7b67950\",\n    \"split\": \"train\"\n  },\n  {\n    \"path\": \"motor_final/data/Motor/NG/bending_1_damage_1_049/A000_I01.png\",\n    \"object\": \"bending_1_damage_1_049\",\n    \"group\": \"049\",\n    \"label\": 1,\n    \"sha256\": \"c2d4630c844071e9a49d837358080b57811f57ddcd5fef516f4160e50e5c7d89\",\n    \"split\": \"train\"\n  }\n]"
CONFIG_TEXT = "{\n  \"created_utc\": \"2026-10-28T08:11:02.215217+00:00\",\n  \"model\": \"Full torchvision ResNet18, ImageNet1K V1 initialization, fc replaced with 512 -> 2\",\n  \"classes\": [\n    \"Normal\",\n    \"Defective\"\n  ],\n  \"regime\": \"Supervised fine-tuning, all parameters trainable\",\n  \"input_size\": 448,\n  \"preprocessing\": \"RGB, full-frame bicubic resize, ImageNet normalization, no augmentation for this engineering pilot\",\n  \"epochs\": 3,\n  \"batch_size\": 4,\n  \"optimizer\": \"AdamW\",\n  \"learning_rate\": 0.0001,\n  \"weight_decay\": 0.0001,\n  \"loss\": \"Unweighted cross entropy\",\n  \"seed\": 20261028,\n  \"threads\": 2,\n  \"decision\": \"argmax of two logits; ties class 0\",\n  \"checkpoint_selection\": \"Highest validation macro-F1, earliest epoch on ties\",\n  \"status\": \"EXPLORATORY ENGINEERING PILOT; old test data reused openly; no independent test\",\n  \"grouping\": \"Conservative numeric suffix grouping, not verified physical identity\",\n  \"split_groups\": {\n    \"train\": 35,\n    \"validation\": 15\n  },\n  \"counts\": {\n    \"train\": {\n      \"0\": 35,\n      \"1\": 35\n    },\n    \"validation\": {\n      \"0\": 15,\n      \"1\": 14\n    }\n  },\n  \"paper_addition\": \"Not yet chosen or implemented; baseline only\",\n  \"future_confirmation\": \"Requires independently held-out data and resolved provenance. No comparison to old anomaly scores as if same training regime.\"\n}"
REVISION_TEXT = "05abc5d03167354a7ef98541236b5c0bd00abadf"
(ROOT / 'resnet18_supervised/manifest.json').write_text(MANIFEST_TEXT)
(ROOT / 'resnet18_supervised/config.json').write_text(CONFIG_TEXT)
(ROOT / 'revision.txt').write_text(REVISION_TEXT)
MANIFEST_SHA256 = 'e32d88936d7a586a8b50691487052b8455b94bbd4c57cb089f4710cf9dc3a145'
CONFIG_SHA256 = '0fd7d77dd821ba99107198e6920c179ff7924e7be16101670b509dc3971fa665'
assert hashlib.sha256(MANIFEST_TEXT.encode()).hexdigest() == MANIFEST_SHA256
assert hashlib.sha256(CONFIG_TEXT.encode()).hexdigest() == CONFIG_SHA256
print('Metadata verified. Torch:',torch.__version__,'Torchvision:',torchvision.__version__)
print('GPU available:',torch.cuda.is_available())


## 2. Secure access to selected original M2AD ZIP images

This uses byte-range requests against the original pinned Motor.zip and verifies every image's original SHA256.
It refuses to silently substitute images or download the full 15 GB ZIP.


In [ ]:
"""Read selected ZIP members with HTTP ranges, validating ZIP CRCs.

Never falls back to downloading the complete archive. Only normal fit/dev
images are selected; held-out test images and calibration images stay closed.
"""
import io,json,urllib.request,zipfile,hashlib,time,struct,zlib
from pathlib import Path

ROOT=Path('/content/nazih_motor_runtime') if Path('/content').is_dir() else Path.cwd() / 'nazih_motor_runtime'
SIZE=14955220087
REV=(ROOT/'revision.txt').read_text().strip()
URL=f'https://huggingface.co/datasets/ChengYuQi99/M2AD/resolve/{REV}/Motor.zip'

class RemoteFile(io.RawIOBase):
    def __init__(self): self.pos=0;self.transferred=0
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self,offset,whence=0):
        self.pos=offset if whence==0 else (self.pos+offset if whence==1 else SIZE+offset)
        if self.pos<0: raise ValueError('Negative seek')
        return self.pos
    def read(self,n=-1):
        n=min(n if n>=0 else SIZE-self.pos,SIZE-self.pos)
        if n<=0:return b''
        if n>20_000_000:raise ValueError(f'Refusing excessive range: {n}')
        start,end=self.pos,self.pos+n-1
        req=urllib.request.Request(URL+f'?range_start={start}&range_end={end}',headers={'Range':f'bytes={start}-{end}'})
        for attempt in range(3):
            try:
                with urllib.request.urlopen(req,timeout=45) as r:
                    if r.status!=206 or r.headers.get('Content-Range')!=f'bytes {start}-{end}/{SIZE}':
                        raise ValueError('Server did not honour exact byte range')
                    b=r.read(n+1)
                if len(b)!=n:raise ValueError('Incomplete range')
                self.pos+=n;self.transferred+=n;return b
            except Exception:
                if attempt==2:raise
                time.sleep(1)

def read_member(remote, info):
    # Local filename/extra-field sizes may differ from central-directory values.
    remote.seek(info.header_offset)
    block=remote.read(min(info.compress_size+131100,SIZE-info.header_offset))
    if block[:4]!=b'PK\x03\x04': raise ValueError('Invalid local ZIP header')
    n,e=struct.unpack_from('<HH',block,26)
    start=30+n+e
    compressed=block[start:start+info.compress_size]
    if len(compressed)!=info.compress_size:raise ValueError('Truncated member')
    if info.compress_type==zipfile.ZIP_DEFLATED: data=zlib.decompress(compressed,-15)
    elif info.compress_type==zipfile.ZIP_STORED:data=compressed
    else:raise ValueError('Unsupported compression')
    if len(data)!=info.file_size or zlib.crc32(data)!=info.CRC:raise ValueError('CRC or size mismatch')
    return data



## 3. Download and verify the exact 99 source images

In [ ]:
"""Recover the *specific* 99 source images in supervised manifest, from the pinned 15-GB ZIP via HTTP byte ranges.
The full 15GB archive is never downloaded. Every file is checked against the SHA256 recorded during the original pilot.
Network access required. The helper follows the original project's known range request format.
"""
import argparse,hashlib,json,zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor,as_completed

ROOT=Path('/content/nazih_motor_runtime') if Path('/content').is_dir() else Path.cwd() / 'nazih_motor_runtime'

def download_dataset():
    workers=4
    assert 1<=workers<=6
    rows=json.loads((ROOT/'resnet18_supervised/manifest.json').read_text())
    pending=[]
    for row in rows:
        dest=ROOT/row['path']
        if dest.exists() and hashlib.sha256(dest.read_bytes()).hexdigest()==row['sha256']:continue
        pending.append(row)
    print(f'Images already verified: {len(rows)-len(pending)}/99; need to retrieve {len(pending)}',flush=True)
    if not pending:return
    with zipfile.ZipFile(RemoteFile()) as z:
        by_suffix={}
        for info in z.infolist():
            name=info.filename.replace('\\','/')
            if name.startswith('Motor/') or '/Motor/' in name:
                by_suffix.setdefault(name[name.index('Motor/'):],[]).append(info)
        jobs=[]
        for row in pending:
            key=row['path'].split('/data/',1)[1]
            candidates=by_suffix.get(key,[])
            if len(candidates)!=1: raise ValueError(f'Expected exactly one ZIP member for {key}, found {len(candidates)}')
            jobs.append((row,candidates[0]))
    def task(job):
        row,info=job
        b=read_member(RemoteFile(),info)
        if hashlib.sha256(b).hexdigest()!=row['sha256']:raise ValueError(f'SHA256 mismatch: {row["path"]}')
        path=ROOT/row['path'];path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(b)
        return row['path']
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for j,fut in enumerate(as_completed([pool.submit(task,j) for j in jobs]),1):
            fut.result()
            if j%10==0 or j==len(jobs):print('Verified',j,'/',len(jobs),'downloaded images',flush=True)
    print('All 99 source-image hashes now match the pilot manifest.')


In [ ]:
download_dataset()


## 4. Model, loss functions, metrics and paired training

The code below is the complete controlled experiment. Both runs start from identical initialization and use the same split and training settings. The comparison is developmental only.


In [ ]:
"""Controlled *development-only* comparison, ResNet18 CE vs Poly-1 CE.
Uses the exact historical supervised pilot manifest, resolution and hyperparameters.
This experiment reuses previously accessed images; validation is NOT a new blind test.
"""
from __future__ import annotations
import argparse,csv,hashlib,json,platform,random,time
from pathlib import Path
import numpy as np
import torch,torchvision
from torch import nn
from torch.nn import functional as F
from PIL import Image
from torchvision.models import resnet18,ResNet18_Weights
ROOT=Path('/content/nazih_motor_runtime') if Path('/content').is_dir() else Path.cwd() / 'nazih_motor_runtime'
MANIFEST=ROOT/'resnet18_supervised/manifest.json'
CONFIG=ROOT/'resnet18_supervised/config.json'
WEIGHTS_SHA='f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec'

def metrics(truth,pred):
    cm=np.zeros((2,2),dtype=int)
    for y,p in zip(truth,pred):cm[int(y),int(p)]+=1
    out={'confusion_matrix':cm.tolist(),'n':int(cm.sum()),'accuracy':float(np.trace(cm)/cm.sum())}
    ps=[];rs=[];fs=[]
    for c in (0,1):
        tp=int(cm[c,c]);fp=int(cm[:,c].sum()-tp);fn=int(cm[c,:].sum()-tp)
        precision=tp/(tp+fp) if tp+fp else 0.;recall=tp/(tp+fn) if tp+fn else 0.
        f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
        ps.append(precision);rs.append(recall);fs.append(f1)
    out.update({'precision_normal':ps[0],'precision_defective':ps[1],
                'recall_normal':rs[0],'recall_defective':rs[1],
                'f1_normal':fs[0],'f1_defective':fs[1],
                'macro_f1':float(np.mean(fs))})
    return out

def poly1_cross_entropy(logits,labels,epsilon=1.0):
    # Eq. in PolyLoss (Leng et al., ICLR 2022): CE + epsilon * (1 - p_true).
    ce=F.cross_entropy(logits,labels,reduction='none')
    true_probability=F.softmax(logits,dim=1).gather(1,labels[:,None]).squeeze(1)
    return (ce + epsilon*(1.-true_probability)).mean()

def load_rows_and_images(cfg):
    rows=json.loads(MANIFEST.read_text())
    assert len(rows)==99 and {r['split'] for r in rows}=={'train','validation'}
    assert {r['label'] for r in rows}=={0,1}
    assert {r['group'] for r in rows if r['split']=='train'}.isdisjoint({r['group'] for r in rows if r['split']=='validation'})
    assert len({r['path'] for r in rows})==len(rows)
    assert {r['path'] for r in rows if r['split']=='train'}.isdisjoint({r['path'] for r in rows if r['split']=='validation'})
    train=[i for i,r in enumerate(rows) if r['split']=='train']
    val=[i for i,r in enumerate(rows) if r['split']=='validation']
    assert len(train)==70 and len(val)==29
    assert {y:sum(rows[i]['label']==y for i in train) for y in (0,1)}=={0:35,1:35}
    assert {y:sum(rows[i]['label']==y for i in val) for y in (0,1)}=={0:15,1:14}
    missing=[r['path'] for r in rows if not (ROOT/r['path']).exists()]
    if missing:raise FileNotFoundError(f'Missing {len(missing)} original images. Run python download_missing.py. Example: {missing[:3]}')
    mean=torch.tensor([.485,.456,.406])[:,None,None];std=torch.tensor([.229,.224,.225])[:,None,None]
    images=[]
    for r in rows:
        path=ROOT/r['path'];b=path.read_bytes()
        if hashlib.sha256(b).hexdigest()!=r['sha256']:raise ValueError(f'Image hash mismatch: {r["path"]}')
        with Image.open(path) as im:
            arr=np.asarray(im.convert('RGB').resize((448,448),Image.Resampling.BICUBIC)).copy()
        image=torch.from_numpy(arr).permute(2,0,1).float()/255
        images.append((image-mean)/std)
    return rows,train,val,images

def set_seed(seed):
    torch.manual_seed(seed);random.seed(seed);np.random.seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)

def model_initial_state(seed):
    set_seed(seed)
    # Keep torchvision construction/order same as historical CE pilot.
    model=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    wp=Path(torch.hub.get_dir())/'checkpoints/resnet18-f37072fd.pth'
    if not wp.exists() or hashlib.sha256(wp.read_bytes()).hexdigest()!=WEIGHTS_SHA:
        raise RuntimeError('Verified ImageNet ResNet18 weights unavailable. Connect to network for one-time download; never substitute random weights.')
    model.fc=nn.Linear(model.fc.in_features,2)
    sha=hashlib.sha256()
    for k,v in model.state_dict().items():sha.update(k.encode());sha.update(v.detach().cpu().numpy().tobytes())
    return model,sha.hexdigest()

def evaluate(model,images,rows,val,batch_size,loss_name,epsilon,device):
    model.eval();truth=[];pred=[];records=[];losssum=0.
    with torch.inference_mode():
        for start in range(0,len(val),batch_size):
            ids=val[start:start+batch_size]
            x=torch.stack([images[i] for i in ids]).to(device)
            y=torch.tensor([rows[i]['label'] for i in ids],device=device)
            z=model(x)
            loss=poly1_cross_entropy(z,y,epsilon) if loss_name=='poly1' else F.cross_entropy(z,y)
            losssum+=loss.item()*len(ids)
            prob=z.softmax(1)
            for k,i in enumerate(ids):
                t=int(y[k]);p=int(z[k].argmax());truth.append(t);pred.append(p)
                records.append({'object':rows[i]['object'],'group':rows[i]['group'],'true_label':t,
                                'predicted_label':p,'prob_defective':float(prob[k,1])})
    stats=metrics(truth,pred);stats['validation_loss']=losssum/len(val)
    return stats,records

def train_one(loss_name,rows,train,val,images,cfg,args,output_dir,device):
    model,initial_hash=model_initial_state(cfg['seed']);model.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg['learning_rate'],weight_decay=cfg['weight_decay'])
    batch_size=cfg['batch_size'];history=[];best=-1.;best_epoch=None;best_rows=None;best_stats=None
    selected_conv_init=model.conv1.weight.detach().clone();selected_fc_init=model.fc.weight.detach().clone()
    beginning=time.perf_counter()
    for epoch in range(1,cfg['epochs']+1):
        model.train();order=torch.randperm(len(train)).tolist();train_loss=0.
        for start in range(0,len(order),batch_size):
            ids=[train[j] for j in order[start:start+batch_size]]
            x=torch.stack([images[i] for i in ids]).to(device)
            y=torch.tensor([rows[i]['label'] for i in ids],device=device)
            opt.zero_grad(set_to_none=True);z=model(x)
            loss=poly1_cross_entropy(z,y,args.epsilon) if loss_name=='poly1' else F.cross_entropy(z,y)
            assert torch.isfinite(loss),'Non-finite loss'
            loss.backward();assert all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in model.parameters())
            opt.step();train_loss+=loss.item()*len(ids)
        stats,per_image=evaluate(model,images,rows,val,batch_size,loss_name,args.epsilon,device)
        history.append({'epoch':epoch,'training_loss':train_loss/len(train),'validation':stats})
        print(loss_name,'epoch',epoch,'val macro-F1',round(stats['macro_f1'],5),flush=True)
        if stats['macro_f1']>best:
            best=stats['macro_f1'];best_epoch=epoch;best_rows=per_image;best_stats=stats
            torch.save(model.state_dict(),output_dir/f'{loss_name}_best.pt')
    updates={'conv1_max_delta':float((model.conv1.weight-selected_conv_init).abs().max()),
             'fc_max_delta':float((model.fc.weight-selected_fc_init).abs().max())}
    assert updates['conv1_max_delta']>0 and updates['fc_max_delta']>0
    with (output_dir/f'{loss_name}_validation_predictions.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(best_rows[0]));writer.writeheader();writer.writerows(best_rows)
    result={'method':loss_name,'initial_state_sha256':initial_hash,'best_epoch':best_epoch,'best_validation':best_stats,
            'training_history':history,'parameter_updates':updates,'train_seconds':time.perf_counter()-beginning}
    (output_dir/f'{loss_name}_result.json').write_text(json.dumps(result,indent=2))
    return result

def run_comparison():
    args=argparse.Namespace(mode='both',epsilon=1.0,device='auto',out='comparison_results')
    assert args.epsilon>=-1 and np.isfinite(args.epsilon)
    cfg=json.loads(CONFIG.read_text())
    assert cfg['epochs']==3 and cfg['batch_size']==4 and cfg['input_size']==448
    assert cfg['seed']==20261028 and cfg['learning_rate']==.0001 and cfg['weight_decay']==.0001
    torch.set_num_threads(cfg['threads'])
    if args.device=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA requested but unavailable')
    device='cuda' if args.device=='auto' and torch.cuda.is_available() else ('cpu' if args.device=='auto' else args.device)
    print('Development validation ONLY. Source data previously accessed; not independent confirmation.',flush=True)
    rows,train,val,images=load_rows_and_images(cfg)
    out=ROOT/args.out;out.mkdir(parents=True,exist_ok=True)
    names=['ce','poly1'] if args.mode=='both' else [args.mode]
    results={}
    for name in names:results[name]=train_one(name,rows,train,val,images,cfg,args,out,device)
    if len(results)==2:
        assert results['ce']['initial_state_sha256']==results['poly1']['initial_state_sha256'], 'Initial weights diverged!'
        ce=results['ce']['best_validation'];po=results['poly1']['best_validation']
        diff={'delta_macro_f1':po['macro_f1']-ce['macro_f1'],
              'delta_defective_recall':po['recall_defective']-ce['recall_defective'],
              'same_initialization':True,
              'claim':'Development comparison only; no independent final test or confirmed specimen identity'}
        (out/'comparison.json').write_text(json.dumps(diff,indent=2))
        print('comparison',diff)
    (out/'run_metadata.json').write_text(json.dumps({'validation_only':True,'old_test_reused':True,
        'verified_physical_independence':False,'seed':cfg['seed'],'epsilon':args.epsilon,
        'environment':{'python':platform.python_version(),'torch':torch.__version__,'torchvision':torchvision.__version__},
        'manifest_sha256':hashlib.sha256(MANIFEST.read_bytes()).hexdigest()},indent=2))


## 5. Safety checks (before any training)

Fail visibly if the manifest split leaks identical images, source objects overlap, or Poly-1 is mathematically inconsistent when epsilon is zero.


In [ ]:
rows=json.loads((ROOT/'resnet18_supervised/manifest.json').read_text())
train_rows=[r for r in rows if r['split']=='train']
validation_rows=[r for r in rows if r['split']=='validation']
assert (len(train_rows),len(validation_rows))==(70,29)
assert {r['group'] for r in train_rows}.isdisjoint({r['group'] for r in validation_rows})
assert {r['path'] for r in train_rows}.isdisjoint({r['path'] for r in validation_rows})
assert {r['sha256'] for r in train_rows}.isdisjoint({r['sha256'] for r in validation_rows})
assert {int(r['label']) for r in rows}=={0,1}
z=torch.tensor([[.4,-.8],[-.1,.3]],requires_grad=True)
y=torch.tensor([0,1])
assert torch.allclose(poly1_cross_entropy(z,y,0.0),F.cross_entropy(z,y),atol=1e-7)
poly1_cross_entropy(z,y,1.0).backward()
assert bool(torch.isfinite(z.grad).all())
print('Pre-training code/split checks passed.')
print('WARNING: Numeric group isolation is not proof of physical part identity independence.')


## 6. Run the two training experiments (GPU recommended)

This cell performs real training. It saves results locally in `/content/nazih_motor_runtime/comparison_results`. **No results for Poly-1 exist until it finishes.**


In [ ]:
run_comparison()


## 7. Show actual comparison and optionally download a results ZIP

Validation numbers from this experiment are *not* an independent blind test. Do not tune against the previously exposed final-test images.


In [ ]:
out=ROOT/'comparison_results'
for name in ('ce','poly1'):
    r=json.loads((out/f'{name}_result.json').read_text())
    print(name.upper(), 'validation:',r['best_validation'])
print('DIFFERENCE:', json.loads((out/'comparison.json').read_text()))
print('LIMITATION: already inspected development data; part identity independence not verified.')


In [ ]:
import shutil
zip_path=shutil.make_archive('/content/NAZIH_Training_Results' if Path('/content').is_dir() else str(ROOT/'NAZIH_Training_Results'),
                              'zip',root_dir=str(ROOT),base_dir='comparison_results')
print('Results ZIP saved locally at:',zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab; take the ZIP from the printed local path.')
